# ANN 12 — Pokémon GAN & VAE
**Deadline:** 07.06.2026  
This notebook covers:
1. Data setup (download, resize, greyscale, normalize)
2. DCGAN (architecture from Radford et al. 2015)
3. VAE with latent space exploration
4. FID evaluation for both models

## 0 — Install & imports

In [10]:
# Run once in Colab to get the Kaggle dataset
# !pip install kaggle torchmetrics[image] --quiet
# from google.colab import files
# files.upload()   # upload your kaggle.json API key
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d kvpratama/pokemon-images-dataset --unzip -p ./pokemon

import os, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils as vutils

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)

Using device: mps


## 1 — Data setup
**Assignment says:** resize to 32×32, convert to greyscale.  
**Why normalize to [-1,1]?** The DCGAN paper uses `Tanh` on the generator output, which has range [-1,1]. Matching the data range to the activation range is critical — if data is [0,1] and the output is Tanh, the model can never perfectly reconstruct the bright pixels, which slows training.

In [11]:
IMG_SIZE  = 32     # assignment requires 32×32
CHANNELS  = 1      # greyscale (set to 3 for colour if you have GPU)
BATCH     = 64
DATA_ROOT = './pokemon'

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.Grayscale(num_output_channels=CHANNELS),  # greyscale as required
    transforms.ToTensor(),
    transforms.Normalize([0.5] * CHANNELS, [0.5] * CHANNELS),  # → [-1, 1]
])

dataset    = datasets.ImageFolder(DATA_ROOT, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH, shuffle=True, num_workers=2)
print(f'Dataset size: {len(dataset)} images')

# Quick sanity check — show a grid of real Pokémon
real_batch, _ = next(iter(dataloader))
plt.figure(figsize=(8,4))
plt.axis('off')
plt.title('Real Pokémon samples')
plt.imshow(vutils.make_grid(real_batch[:32], nrow=8, normalize=True).permute(1,2,0).cpu())

FileNotFoundError: [Errno 2] No such file or directory: './pokemon'

## 2 — DCGAN architecture
### Why these specific choices? (straight from the paper)
| Choice | Paper rule | Reason |
|---|---|---|
| `ConvTranspose2d` with stride 2 | Replace pooling with strided conv | Network learns its own upsampling |
| `BatchNorm2d` on all but first/last layers | BN everywhere except G output and D input | Prevents mode collapse; stabilizes gradients |
| `ReLU` in G, `LeakyReLU(0.2)` in D | Paper Table 1 | ReLU in G saturates less; LeakyReLU in D keeps gradients alive for negative activations |
| `Tanh` on G output | Bounded activation | Matches data range [-1,1]; model learns colour distribution faster |
| No fully-connected hidden layers | Remove FC layers | Fewer parameters, more spatial structure |
| Adam lr=0.0002, β1=0.5 | Paper §4 | lr=0.001 caused oscillation; β1=0.9 caused instability |

In [ ]:
LATENT_DIM = 100   # z noise vector size (standard in DCGAN paper)
NGF        = 64    # generator feature map base (doubles each layer going backwards)
NDF        = 64    # discriminator feature map base

def weights_init(m):
    """DCGAN paper §4: init weights from N(0, 0.02)"""
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# ── Generator ────────────────────────────────────────────────────────
# Input:  z of shape (batch, 100, 1, 1)
# Output: image of shape (batch, 1, 32, 32)
# Each ConvTranspose2d with stride=2 doubles the spatial size.
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # z: (100,1,1) → (NGF*4, 4, 4)
            nn.ConvTranspose2d(LATENT_DIM, NGF*4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF*4),
            nn.ReLU(True),
            # (NGF*4,4,4) → (NGF*2, 8, 8)
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*2),
            nn.ReLU(True),
            # (NGF*2,8,8) → (NGF, 16,16)
            nn.ConvTranspose2d(NGF*2, NGF, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF),
            nn.ReLU(True),
            # (NGF,16,16) → (C, 32,32)  — NO BatchNorm on final layer!
            nn.ConvTranspose2d(NGF, CHANNELS, 4, 2, 1, bias=False),
            nn.Tanh()   # output in [-1,1] to match normalized data
        )
    def forward(self, z): return self.net(z)

# ── Discriminator ─────────────────────────────────────────────────────
# Mirror of generator, strided conv to downsample.
# LeakyReLU(0.2) keeps gradients flowing even for 'fake' predictions.
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # Input: (C,32,32) — NO BatchNorm on first layer!
            nn.Conv2d(CHANNELS, NDF, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # (NDF,16,16)
            nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*2),
            nn.LeakyReLU(0.2, inplace=True),
            # (NDF*2,8,8)
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*4),
            nn.LeakyReLU(0.2, inplace=True),
            # (NDF*4,4,4) → scalar
            nn.Conv2d(NDF*4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()   # output probability: real=1, fake=0
        )
    def forward(self, x): return self.net(x).view(-1)

G = Generator().to(DEVICE).apply(weights_init)
D = Discriminator().to(DEVICE).apply(weights_init)
print(G)
print(D)

## 3 — GAN training loop
**The minimax game:**
```
Loss_D = -[ log D(x_real) + log(1 - D(G(z))) ]
Loss_G = -[ log D(G(z)) ]   ← non-saturating trick from the paper
```
The non-saturating trick for G (`log D(G(z))` instead of `log(1-D(G(z)))`) is important early in training when D easily rejects fakes — the gradient would otherwise be nearly zero, meaning G learns nothing.

In [ ]:
criterion = nn.BCELoss()
# Paper: lr=0.0002, β1=0.5 (β1=0.9 caused oscillation)
opt_D = Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_G = Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Fixed noise to track training progress visually
fixed_z = torch.randn(64, LATENT_DIM, 1, 1, device=DEVICE)

EPOCHS     = 50
G_losses, D_losses = [], []
img_list   = []

for epoch in range(EPOCHS):
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(DEVICE)
        bs        = real_imgs.size(0)

        # ── Train Discriminator ──────────────────────────────────────
        # Goal: D(real)→1, D(fake)→0
        D.zero_grad()

        # Real images: label = 1
        real_labels = torch.ones(bs, device=DEVICE)
        out_real    = D(real_imgs)
        loss_D_real = criterion(out_real, real_labels)

        # Fake images: label = 0
        z           = torch.randn(bs, LATENT_DIM, 1, 1, device=DEVICE)
        fake_imgs   = G(z)
        fake_labels = torch.zeros(bs, device=DEVICE)
        out_fake    = D(fake_imgs.detach())  # detach: don't backprop into G here
        loss_D_fake = criterion(out_fake, fake_labels)

        loss_D = loss_D_real + loss_D_fake
        loss_D.backward()
        opt_D.step()

        # ── Train Generator ──────────────────────────────────────────
        # Goal: fool D → D(G(z))→1
        # Non-saturating loss: maximize log D(G(z)) instead of minimizing log(1-D(G(z)))
        G.zero_grad()
        out_fake2 = D(fake_imgs)   # re-evaluate (graph still alive)
        loss_G    = criterion(out_fake2, real_labels)  # G wants D to call fakes 'real'
        loss_G.backward()
        opt_G.step()

        G_losses.append(loss_G.item())
        D_losses.append(loss_D.item())

    # Save generated grid every 5 epochs
    if (epoch+1) % 5 == 0:
        with torch.no_grad():
            fake = G(fixed_z).cpu()
        img_list.append(vutils.make_grid(fake, nrow=8, normalize=True))
        print(f'Epoch [{epoch+1}/{EPOCHS}]  Loss_D={loss_D:.4f}  Loss_G={loss_G:.4f}')

# Save final generated images for submission
os.makedirs('generated', exist_ok=True)
for i, grid in enumerate(img_list):
    vutils.save_image(grid, f'generated/gan_epoch_{(i+1)*5:03d}.png')

# Plot losses
plt.figure(figsize=(10,4))
plt.plot(G_losses, label='Generator loss', alpha=0.6)
plt.plot(D_losses, label='Discriminator loss', alpha=0.6)
plt.xlabel('Iteration'); plt.ylabel('Loss'); plt.legend()
plt.title('GAN training losses')
plt.tight_layout(); plt.savefig('generated/gan_losses.png', dpi=120); plt.show()

## 4 — Variational Autoencoder
**Key difference from GAN:**  
- GAN learns by adversarial feedback (no explicit loss on pixels)  
- VAE optimizes a tractable lower bound on the data likelihood (ELBO):
```
ELBO = E[log p(x|z)] - KL(q(z|x) || p(z))
     = Reconstruction loss + KL divergence
```
The KL term forces the latent space to be a smooth Gaussian, which is why you can interpolate between Pokémon by moving in z-space.

**Convolutional VAE** — same idea as the simple VAE from earlier sessions, but with Conv layers instead of FC, so it respects spatial structure better on images.

In [ ]:
LATENT_VAE = 64   # try: 8, 32, 64, 128 — assignment asks you to compare!

class ConvVAE(nn.Module):
    def __init__(self, latent_dim=LATENT_VAE):
        super().__init__()
        self.latent_dim = latent_dim

        # ── Encoder: image → (μ, logσ²) ───────────────────────────
        self.encoder = nn.Sequential(
            nn.Conv2d(CHANNELS, 32, 4, 2, 1),   # (C,32,32)→(32,16,16)
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),          # (32,16,16)→(64,8,8)
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),         # (64,8,8)→(128,4,4)
            nn.ReLU(),
            nn.Flatten()                          # → 128*4*4 = 2048
        )
        self.fc_mu     = nn.Linear(2048, latent_dim)
        self.fc_logvar = nn.Linear(2048, latent_dim)

        # ── Decoder: z → image ────────────────────────────────────
        self.fc_decode = nn.Linear(latent_dim, 2048)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, CHANNELS, 4, 2, 1),
            nn.Sigmoid()   # output in [0,1] — matches BCE loss
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        """The reparameterization trick: z = μ + ε·σ, ε~N(0,1)
        Allows gradients to flow through the sampling operation."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_decode(z).view(-1, 128, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    """ELBO loss = reconstruction + beta * KL divergence.
    beta=1 is standard VAE; beta>1 is beta-VAE (more disentangled latent space).
    Data must be in [0,1] for BCE to work.
    """
    # Normalize from [-1,1] back to [0,1] for BCE
    x_01     = (x + 1) / 2
    recon    = F.binary_cross_entropy(x_hat, x_01, reduction='sum')
    # KL = -0.5 * sum(1 + logσ² - μ² - σ²)  — closed form for Gaussian prior
    kl       = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl

vae     = ConvVAE(latent_dim=LATENT_VAE).to(DEVICE)
opt_vae = Adam(vae.parameters(), lr=1e-3)
print(vae)

In [ ]:
VAE_EPOCHS = 50
vae_losses = []

for epoch in range(VAE_EPOCHS):
    vae.train()
    total_loss = 0
    for imgs, _ in dataloader:
        imgs = imgs.to(DEVICE)
        opt_vae.zero_grad()
        x_hat, mu, logvar = vae(imgs)
        loss = vae_loss(x_hat, imgs, mu, logvar)
        loss.backward()
        opt_vae.step()
        total_loss += loss.item()

    avg = total_loss / len(dataset)
    vae_losses.append(avg)
    if (epoch+1) % 10 == 0:
        print(f'VAE Epoch [{epoch+1}/{VAE_EPOCHS}]  Loss={avg:.2f}')

# Show reconstructions
vae.eval()
sample_imgs, _ = next(iter(dataloader))
sample_imgs = sample_imgs[:16].to(DEVICE)
with torch.no_grad():
    recons, _, _ = vae(sample_imgs)
comparison = torch.cat([sample_imgs.cpu(), recons.cpu() * 2 - 1])  # both to [-1,1]
plt.figure(figsize=(10,4))
plt.axis('off')
plt.title('Top: real | Bottom: VAE reconstruction')
plt.imshow(vutils.make_grid(comparison, nrow=16, normalize=True).permute(1,2,0))
plt.savefig('generated/vae_reconstructions.png', dpi=120); plt.show()

## 5 — Latent space exploration
**Assignment question:** *Is there meaningful/interesting structure in the latent space?*

Three things to check:
1. **Interpolation** — smoothly blend between two Pokémon in z-space. If the VAE learned meaningful representations, the in-between images should look like plausible intermediate Pokémon, not random noise.
2. **t-SNE plot** — project all encoded Pokémon to 2D. Clusters = types that look similar to the model.
3. **Latent dim sweep** — retrain with dim=8, 32, 128. Smaller dim = more compression = blurrier but more regular; larger = sharper but more fragmented space.

In [ ]:
# ── 5a: Interpolation between two Pokémon ──────────────────────────
vae.eval()
imgs_batch, _ = next(iter(dataloader))
img_a = imgs_batch[0:1].to(DEVICE)   # Pokémon A
img_b = imgs_batch[1:2].to(DEVICE)   # Pokémon B

with torch.no_grad():
    mu_a, logvar_a = vae.encode(img_a)
    mu_b, logvar_b = vae.encode(img_b)
    z_a = vae.reparameterize(mu_a, logvar_a)
    z_b = vae.reparameterize(mu_b, logvar_b)

steps = 10
interps = []
for alpha in np.linspace(0, 1, steps):
    z_interp = (1 - alpha) * z_a + alpha * z_b   # linear interpolation in z-space
    with torch.no_grad():
        decoded = vae.decode(z_interp).cpu()
    interps.append(decoded)

interps = torch.cat(interps)
plt.figure(figsize=(12, 2))
plt.axis('off')
plt.title('Latent space interpolation: Pokémon A → Pokémon B')
plt.imshow(vutils.make_grid(interps, nrow=steps, normalize=True).permute(1,2,0))
plt.savefig('generated/vae_interpolation.png', dpi=120); plt.show()

In [ ]:
# ── 5b: t-SNE of latent codes ──────────────────────────────────────
from sklearn.manifold import TSNE

all_mu = []
vae.eval()
with torch.no_grad():
    for imgs, _ in dataloader:
        mu, _ = vae.encode(imgs.to(DEVICE))
        all_mu.append(mu.cpu().numpy())

all_mu = np.concatenate(all_mu, axis=0)
tsne   = TSNE(n_components=2, perplexity=30, random_state=SEED)
z_2d   = tsne.fit_transform(all_mu)

plt.figure(figsize=(7,7))
plt.scatter(z_2d[:,0], z_2d[:,1], s=8, alpha=0.5)
plt.title(f'VAE latent space (t-SNE), latent_dim={LATENT_VAE}')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.savefig('generated/vae_tsne.png', dpi=120); plt.show()

In [ ]:
# ── 5c: Generate new Pokémon by sampling from prior ─────────────────
# Sample z ~ N(0,1) directly (the prior), decode to image
# This only works well if KL term successfully pushed the posterior towards N(0,1)
z_sample = torch.randn(64, LATENT_VAE, device=DEVICE)
with torch.no_grad():
    gen_pokemon = vae.decode(z_sample).cpu()

plt.figure(figsize=(8,4))
plt.axis('off')
plt.title('VAE generated Pokémon (z ~ N(0,1))')
plt.imshow(vutils.make_grid(gen_pokemon, nrow=8, normalize=True).permute(1,2,0))
plt.savefig('generated/vae_generated.png', dpi=120); plt.show()

## 6 — FID Score
**What FID measures:**  
FID (Fréchet Inception Distance) computes the distance between the distribution of real images and generated images in the feature space of InceptionV3.
```
FID = ||μ_r - μ_g||² + Tr(Σ_r + Σ_g - 2(Σ_r Σ_g)^0.5)
```
- Lower FID = generated images look more like real ones
- You need at least 1000 generated samples for a reliable estimate
- GAN typically has lower FID (sharper), VAE has higher FID (blurrier) — this is the core tradeoff

**Caveat:** InceptionV3 was trained on colour 299×299 ImageNet images. Our 32×32 greyscale Pokémon will be upscaled and converted to 3-channel, so the FID values should only be compared *between* your models, not against published benchmarks.

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance

def compute_fid(real_loader, generator_fn, n_samples=1000, device=DEVICE):
    """Compute FID between real dataset and samples from generator_fn.
    generator_fn() should return a batch of images in [0,255] uint8 format.
    """
    fid = FrechetInceptionDistance(feature=2048).to(device)

    # Feed real images (FID needs uint8 RGB)
    print('Computing FID: adding real images...')
    count = 0
    for imgs, _ in real_loader:
        if count >= n_samples: break
        imgs_rgb  = imgs.repeat(1, 3, 1, 1)                  # grey → 3-channel
        imgs_uint = ((imgs_rgb + 1) / 2 * 255).byte().to(device)
        fid.update(imgs_uint, real=True)
        count += imgs.size(0)

    # Feed generated images
    print('Computing FID: adding generated images...')
    count = 0
    while count < n_samples:
        batch = generator_fn(min(64, n_samples - count))
        batch_rgb  = batch.repeat(1, 3, 1, 1)
        batch_uint = (batch_rgb * 255).byte().to(device)
        fid.update(batch_uint, real=False)
        count += batch.size(0)

    score = fid.compute().item()
    print(f'FID = {score:.2f}')
    return score

# ── FID for GAN ─────────────────────────────────────────────────────
G.eval()
def gan_generator(n):
    z = torch.randn(n, LATENT_DIM, 1, 1, device=DEVICE)
    with torch.no_grad():
        imgs = G(z)
    return ((imgs + 1) / 2).cpu()   # to [0,1]

fid_gan = compute_fid(dataloader, gan_generator, n_samples=1000)

# ── FID for VAE ─────────────────────────────────────────────────────
vae.eval()
def vae_generator(n):
    z = torch.randn(n, LATENT_VAE, device=DEVICE)
    with torch.no_grad():
        imgs = vae.decode(z)
    return imgs.cpu()   # already in [0,1]

fid_vae = compute_fid(dataloader, vae_generator, n_samples=1000)

print(f'\n=== FID Results ===')
print(f'GAN FID: {fid_gan:.2f}')
print(f'VAE FID: {fid_vae:.2f}')
print('Lower = better (closer to real data distribution)')

## 7 — Summary & analysis
Fill this in based on your results:

### GAN observations
- Loss curve: did D and G reach a reasonable equilibrium?
- Visual quality: are the generated Pokémon recognizable as sprites?
- Mode collapse: do all samples look the same? (if so, try training D less frequently)

### VAE observations
- Interpolation: smooth or jumpy? Smooth = KL term worked; jumpy = too little KL weight
- t-SNE: any clusters? What might they encode (colour, shape, size?)
- Latent dim comparison:

| latent_dim | FID | Notes |
|---|---|---|
| 8 | ? | Very compressed, smooth space |
| 32 | ? | Balanced |
| 64 | ? | Default run above |
| 128 | ? | Most detail, but less regular space |

### GAN vs VAE comparison
- GAN FID: **{fid_gan}**  
- VAE FID: **{fid_vae}**  
- GAN tends to have lower FID (sharper images) but can mode collapse; VAE is stable but blurry. This is the classic GAN/VAE tradeoff.

In [ ]:
# Save models for later
torch.save(G.state_dict(), 'generated/generator.pth')
torch.save(D.state_dict(), 'generated/discriminator.pth')
torch.save(vae.state_dict(), 'generated/vae.pth')
print('Models saved. Upload this notebook and the generated/ folder to StudIP.')